In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

_DATA_DIR     = "/teamspace/studios/this_studio"
_OUTPUT_DIR   = "/teamspace/studios/this_studio"
_ARTIFACT_DIR = "/teamspace/studios/this_studio/artifacts"

CFG = dict(
    SEED            = 42,
    N_FOLDS         = 4,
    EPOCHS_DEB      = 2,
    BS_DEB          = 1,
    GRAD_ACCUM_DEB  = 4,
    MAX_LEN_DEB     = 512,
    LR_DEB          = 1e-5,
    MARGIN          = 0.5,
    USE_BF16        = True,
    DATA_DIR        = _DATA_DIR,
    OUTPUT_DIR      = _OUTPUT_DIR,
    ARTIFACT_DIR    = _ARTIFACT_DIR,
    DEBERTA_MODEL   = 'microsoft/deberta-v3-large',
    RUN_DEBERTA     = True,
)

os.makedirs(CFG['OUTPUT_DIR'],   exist_ok=True)
os.makedirs(CFG['ARTIFACT_DIR'], exist_ok=True)

OPTS = ['A', 'B', 'C', 'D', 'E']

CFG['RUN_RAG']         = True
CFG['RAG_EMBED_MODEL'] = 'BAAI/bge-large-en-v1.5'
CFG['RAG_PER_SOURCE_CAP'] = 60000
CFG['RAG_TOP_K']    = 3
CFG['RAG_CTX_CHARS']= 400
CFG['WIKI_CONFIG']  = '20231101.simple'

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
import os
import kagglehub
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast as _autocast, GradScaler as _GradScaler
from sentence_transformers import SentenceTransformer

import faiss
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.neighbors import NearestNeighbors


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_all(CFG['SEED'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute device : {DEVICE.type.upper()}")

Compute device: CUDA
GPU: NIVIDIA A100-SXM4-40GB


In [34]:
import os
try:
    import wandb
    WANDB = wandb
except ImportError:
    wandb = None
    WANDB = None

USE_WANDB     = True
RUN_VERSION   = 1
RUN_NOTES     = "r_deberta_v6, n_folds=4, epoch=2"
WANDB_PROJECT = "24f1000134-t22026"

if USE_WANDB and wandb is not None:
    try:
        wandb.login(key=os.environ.get("WANDB_API_KEY"))
        print("W&B connected.")
    except Exception as e:
        print(f"W&B login skipped ({e}).")
        USE_WANDB = False
elif wandb is None:
    USE_WANDB = False
    print("wandb not installed; logging disabled.")

def wandb_init(model_tag, fold):
    if not (USE_WANDB and wandb is not None):
        return None
    return wandb.init(
        project=WANDB_PROJECT,
        name=f"{model_tag}_v{RUN_VERSION}_fold{fold}",
        group=f"{model_tag}_v{RUN_VERSION}",
        job_type="train",
        notes=RUN_NOTES,
        tags=[model_tag, f"v{RUN_VERSION}"],
        config={**CFG, "run_version": RUN_VERSION,
                "run_notes": RUN_NOTES, "model": model_tag, "fold": fold},
        reinit=True,
    )

def wlog(metrics, step=None):
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.log(metrics, step=step)

def wandb_finish():
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /teamspace/studios/this_studio/.netrc


W&B connected.


In [36]:
KAGGLE_USERNAME = "kamal134134"   # KAGGLE: your Kaggle username
PUSH_TO_HUB     = True
HUB_FRAMEWORK   = "pyTorch"

HUB_HANDLES = {
    
    "deberta": f"{KAGGLE_USERNAME}/genai_project/{HUB_FRAMEWORK}/deberta_rag_l_4_2"
}

def push_model(model_tag, folder, notes=""):
    if not PUSH_TO_HUB or kagglehub is None:
        print(f"  [hub] upload skipped for '{model_tag}'")
        return
    handle = HUB_HANDLES[model_tag]
    note = notes or f"v{RUN_VERSION}: {RUN_NOTES}"
    print(f"  [hub] uploading '{model_tag}' ...")
    
    
    kagglehub.model_upload(handle, folder, version_notes=note)
    print("  [hub] upload complete")

In [37]:
os.environ['HF_TOKEN'] = os.environ.get('HF_TOKEN', 'PASTE_HF_TOKEN_HERE')

print("Preparing DeBERTa-v3-large (tokenizer + weights) ...")
t = time.time()
_tok = AutoTokenizer.from_pretrained(CFG['DEBERTA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  tokenizer ready   ({time.time()-t:.0f}s)")
t = time.time()
_mdl = AutoModel.from_pretrained(CFG['DEBERTA_MODEL'], token=os.environ['HF_TOKEN'])
print(f"  model weights ready ({time.time()-t:.0f}s, ~1.7 GB)")
del _mdl, _tok; gc.collect(); torch.cuda.empty_cache()
print("  model cached — ready to train")

Preparing DeBERTa-v3-large (tokenizer + weights) ...
  tokenizer ready   (0s)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  model weights ready (1s, ~1.7 GB)
  model cached — ready to train


In [7]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
print(f"Data loaded  train = {train.shape[0]} rows,  test = {test.shape[0]} rows")
print(f"Columns   {list(train.columns)}")

Data loaded  train = 2000 rows,  test = 500 rows
Columns   ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [8]:
from sklearn.model_selection import GroupKFold

_BOILERPLATE_PAT = re.compile(
    r"pick the best possible answer:?|\bcarefully\b\.?|among the listed options\.?|"
    r"choose the (correct|best) (option|answer|response)\.?|"
    r"select the (best|correct|most appropriate) (option|answer|response)\.?|"
    r"determine the correct option:?|identify the correct statement:?|"
    r"based on the given context\.?|which of the following( statements)?( is true)?",
    flags=re.IGNORECASE
)
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = _BOILERPLATE_PAT.sub(' ', str(text))
    return _WS.sub(' ', s).strip()

# Apply normalization
for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)

# GroupKFold assignment
gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.iloc[va_idx, train.columns.get_loc('fold')] = fold_id

# Leakage Checks
assert (train['fold'] >= 0).all() and train.groupby('prompt_n')['fold'].nunique().max() == 1, "Leak detected!"
print("Folds:\n", train['fold'].value_counts().sort_index().to_string())

Folds:
 fold
0    500
1    500
2    500
3    500


In [ ]:
CAP = CFG["RAG_PER_SOURCE_CAP"]
CONTEXT_MAP = {}

def chunk_text(text, chars=500, overlap=100):
    text = text or ""
    chunks = []
    start = 0

    while start < len(text):
        piece = text[start:start + chars].strip()
        if len(piece) > 100:
            chunks.append(piece)
        start += chars - overlap

    return chunks


def load_stream_dataset(name, config=None):
    passages = []

    ds = load_dataset(
        name,
        config,
        split="train",
        streaming=True,
    )

    field = None

    for row in ds:

        if field is None:
            field = next(
                (f for f in ("text", "content", "article", "passage") if f in row),
                None,
            )

        passages.extend(chunk_text(str(row[field])))

        if len(passages) >= CAP:
            break

    return passages[:CAP]


print("[RAG] Building corpus...")
t0 = time.time()

# Wikipedia
wiki = load_stream_dataset(
    "wikimedia/wikipedia",
    CFG["WIKI_CONFIG"],
)

print(f"[Wikipedia] {len(wiki)} passages")

# SEP
sep = load_stream_dataset("AiresPucrs/stanford-encyclopedia-philosophy")

print(f"[SEP] {len(sep)} passages")

passages = list(dict.fromkeys(wiki + sep))

print(
    f"[RAG] Total corpus: {len(passages)} passages "
    f"(loaded in {time.time() - t0:.1f}s)"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = SentenceTransformer(
    CFG["RAG_EMBED_MODEL"],
    device=device,
)

print("[RAG] Encoding corpus...")

corpus_emb = encoder.encode(
    passages,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_prefix = "Represent this sentence for searching relevant passages: "

questions = (
    test["prompt"].tolist()
    + train["prompt"].tolist()
)

question_ids = (
    test["id"].tolist()
    + train["id"].tolist()
)

print("[RAG] Encoding questions...")

question_emb = encoder.encode(
    [query_prefix + q for q in questions],
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

top_k = CFG["RAG_TOP_K"]


index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb.astype("float32"))

_, neighbors = index.search(
    question_emb.astype("float32"),
    top_k,
)

print("[RAG] Retrieval via FAISS")

limit = CFG["RAG_CTX_CHARS"]

for qid, idx in zip(question_ids, neighbors):
    CONTEXT_MAP[qid] = (
        " ".join(passages[i] for i in idx)
    )[:limit]

print(f"[RAG] CONTEXT_MAP built for {len(CONTEXT_MAP)} questions")
print(CONTEXT_MAP[test["id"].iloc[0]][:220], "...")

del corpus_emb, question_emb, encoder

gc.collect()
torch.cuda.empty_cache()

[RAG] Building multi-source corpus ...
  [wikipedia] 60001 passages
  [sep] 60000 passages (from AiresPucrs/stanford-encyclopedia-philosophy)
[RAG] Total corpus: 119699 passages (load time 3s)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[RAG] Encoding corpus (this is the self-built index) ...


Batches:   0%|          | 0/468 [00:00<?, ?it/s]

[RAG] Encoding questions ...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

[RAG] Retrieval via self-built FAISS index
[RAG] CONTEXT_MAP built for 2000 questions
[RAG] Example retrieved context (first test question):
    Heidegger’s conception of human existence (or, as calls it, Dasein, ‘being-there’) echoes Kierkegaard’s conception of the “self”. Rather than being an object among others, Dasein is a “relation of being” (Seinsverhältnis ...


In [41]:
def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = order[i, :3]
        if y_idx[i] in top3:
            total += 1.0 / (list(top3).index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx     = np.asarray(y_idx)
    pred_top1 = scores_2d.argmax(axis=1)
    return dict(
        map3 = map_at_3(scores_2d, y_idx),
        acc  = accuracy_score(y_idx, pred_top1),
        f1   = f1_score(y_idx, pred_top1, average='macro'),
    )

def pairwise_margin_loss(scores, correct, margin=0.5):
    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss); mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)

In [42]:
class DebertaScorer(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        h = self.backbone.config.hidden_size
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(h, h // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(h // 2, 1),
        )

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls).squeeze(-1)

class DebertaDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, context_map=None,
                  shuffle_opts=False, has_label=True, seed=0):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len
        self.ctx = context_map or {}
        self.shuffle = shuffle_opts
        self.has_label = has_label
        self.rng = np.random.default_rng(seed)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        prompt = r['prompt_n']
        if r['id'] in self.ctx:
            prompt = self.ctx[r['id']] + ' ' + prompt
        options = [r[f'{o}_n'] for o in OPTS]
        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            options = [options[i] for i in perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])

        enc = self.tok([prompt]*5, options, truncation=True,
                       padding='max_length', max_length=self.max_len,
                       return_tensors='pt')
        return dict(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            correct=torch.tensor(correct, dtype=torch.long),
            id=int(r['id']),
        )

In [43]:
def train_deberta_cv(train, test_df, n_folds=CFG['N_FOLDS']):
    MODEL_TAG  = 'deberta'
    artifact_dir = f"{CFG['ARTIFACT_DIR']}/{MODEL_TAG}"
    os.makedirs(artifact_dir, exist_ok=True)

    oof_scores  = np.zeros((len(train), 5), dtype=np.float32)
    test_scores = np.zeros((len(test_df),  5), dtype=np.float32)
    y_idx = np.array([OPTS.index(a) for a in train['answer']])

    tokenizer = AutoTokenizer.from_pretrained(CFG['DEBERTA_MODEL'])

    assert 'fold' in train.columns, "train must have a 'fold' column (set in Part 2)"

    for fold in range(n_folds):
        run = wandb_init(MODEL_TAG, fold)

        va_idx = np.where(train['fold'].values == fold)[0]
        tr_idx = np.where(train['fold'].values != fold)[0]
        print(f'\n[DeBERTa] === fold {fold} (train={len(tr_idx)}, val={len(va_idx)}) ===', flush=True)

        tr_ds = DebertaDataset(train.iloc[tr_idx], tokenizer, CFG['MAX_LEN_DEB'],
                                CONTEXT_MAP, shuffle_opts=True,  seed=fold)
        va_ds = DebertaDataset(train.iloc[va_idx], tokenizer, CFG['MAX_LEN_DEB'],
                                CONTEXT_MAP, shuffle_opts=False)
        te_ds = DebertaDataset(test_df, tokenizer, CFG['MAX_LEN_DEB'],
                                CONTEXT_MAP, shuffle_opts=False, has_label=False)

        tr_dl = DataLoader(tr_ds, batch_size=CFG['BS_DEB'], shuffle=True,
                           num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, batch_size=CFG['BS_DEB'], shuffle=False,
                           num_workers=0, pin_memory=True)
        te_dl = DataLoader(te_ds, batch_size=CFG['BS_DEB'], shuffle=False,
                           num_workers=0, pin_memory=True)

        model = DebertaScorer(CFG['DEBERTA_MODEL']).float().to(DEVICE)

        optim = AdamW(model.parameters(), lr=CFG['LR_DEB'], weight_decay=0.01)
        steps = CFG['EPOCHS_DEB'] * len(tr_dl) // CFG['GRAD_ACCUM_DEB']
        sched = CosineAnnealingLR(optim, T_max=max(steps, 1))
        accum = CFG['GRAD_ACCUM_DEB']

        best_val_map3, best_va_scores, best_te_scores = -1, None, None

        for epoch in range(CFG['EPOCHS_DEB']):
            t_epoch = time.time()
            model.train()
            train_loss, n_train = 0.0, 0
            optim.zero_grad()

            for step, batch in enumerate(tr_dl):
                B = batch['input_ids'].size(0)
                ids  = batch['input_ids'].view(B*5, -1).to(DEVICE, non_blocking=True)
                mask = batch['attention_mask'].view(B*5, -1).to(DEVICE, non_blocking=True)
                correct = batch['correct'].to(DEVICE, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", dtype=torch.float32, enabled=False):
                    scores = model(ids, mask).view(B, 5)
                    loss = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN']) / accum

                if torch.isfinite(loss):
                    loss.backward()
                else:
                    print(f'    [warn] step {step}: non-finite loss, skipping backward')

                if (step + 1) % accum == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optim.step(); sched.step()
                    optim.zero_grad()

                train_loss += loss.item() * B * accum
                n_train += B

                if step % 25 == 0:
                    print(f'    step {step}/{len(tr_dl)}  '
                          f'loss={loss.item()*accum:.4f}  '
                          f'elapsed={time.time()-t_epoch:.0f}s', flush=True)

            model.eval()
            va_loss, n_val = 0.0, 0
            va_scores_all = []
            with torch.no_grad():
                for batch in va_dl:
                    B = batch['input_ids'].size(0)
                    ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                    mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                    correct = batch['correct'].to(DEVICE)
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float32, enabled=False):
                        scores = model(ids, mask).view(B, 5)
                        loss = pairwise_margin_loss(scores, correct, margin=CFG['MARGIN'])
                    va_loss += loss.item() * B
                    n_val += B
                    va_scores_all.append(scores.cpu().float().numpy())
            va_scores_all = np.concatenate(va_scores_all)
            val_m = all_metrics(va_scores_all, y_idx[va_idx])

            tr_loss_val = train_loss / max(n_train, 1)
            va_loss_val = va_loss / max(n_val, 1)

            print(f'  ep{epoch} ({time.time()-t_epoch:.0f}s)  '
                  f'train_loss={tr_loss_val:.4f} '
                  f'val_loss={va_loss_val:.4f} val_map3={val_m["map3"]:.4f} '
                  f'val_acc={val_m["acc"]:.4f} val_f1={val_m["f1"]:.4f}', flush=True)

            wlog({
                'deb/train_loss': tr_loss_val,
                'deb/val_loss':   va_loss_val,
                'deb/val_map3':   val_m['map3'],
                'deb/val_acc':    val_m['acc'],
                'deb/val_f1':     val_m['f1'],
                'deb/fold':       fold,
                'deb/epoch':      epoch,
            })

            if val_m['map3'] > best_val_map3:
                best_val_map3  = val_m['map3']
                best_va_scores = va_scores_all
                model.eval()
                te_scores = []
                with torch.no_grad():
                    for batch in te_dl:
                        B = batch['input_ids'].size(0)
                        ids  = batch['input_ids'].view(B*5, -1).to(DEVICE)
                        mask = batch['attention_mask'].view(B*5, -1).to(DEVICE)
                        with torch.amp.autocast(device_type="cuda", dtype=torch.float32, enabled=False):
                            scores = model(ids, mask).view(B, 5)
                        te_scores.append(scores.cpu().float().numpy())
                best_te_scores = np.concatenate(te_scores)

                torch.save(model.state_dict(), f"{artifact_dir}/best_deberta_fold{fold}.pth")
                print(f"    -> new best (val_map3={best_val_map3:.4f}), checkpoint saved", flush=True)

        wlog({'deb/best_val_map3': best_val_map3})
        oof_scores[va_idx] = best_va_scores
        test_scores += best_te_scores / n_folds
        wandb_finish()
        del model; gc.collect(); torch.cuda.empty_cache()

    overall = all_metrics(oof_scores, y_idx)
    print(f'\n[DeBERTa] OOF MAP@3={overall["map3"]:.4f} '
          f'acc={overall["acc"]:.4f} f1={overall["f1"]:.4f}', flush=True)

    if USE_WANDB and wandb is not None:
        summary_run = wandb.init(
            project=WANDB_PROJECT,
            name=f"{MODEL_TAG}_v{RUN_VERSION}_OOF",
            group=f"{MODEL_TAG}_v{RUN_VERSION}",
            job_type="summary", tags=[MODEL_TAG, f"v{RUN_VERSION}", "oof"],
            reinit=True,
        )
        wandb.log({
            'deb/oof_map3': overall['map3'],
            'deb/oof_acc':  overall['acc'],
            'deb/oof_f1':   overall['f1'],
        })
        wandb.run.summary['oof_map3'] = overall['map3']
        wandb.run.summary['oof_acc']  = overall['acc']
        wandb.run.summary['oof_f1']   = overall['f1']
        wandb.finish()

    np.save(f"{artifact_dir}/oof_deb.npy",  oof_scores)
    np.save(f"{artifact_dir}/test_deb.npy", test_scores)
    print(f"Saved oof_deb.npy + test_deb.npy in {artifact_dir}")

    push_model(
        MODEL_TAG,
        artifact_dir,
        notes=f"v{RUN_VERSION}: {RUN_NOTES} | OOF MAP@3={overall['map3']:.4f} "
              f"acc={overall['acc']:.4f} f1={overall['f1']:.4f}"
    )

    return oof_scores, test_scores




oof_deb, test_deb = train_deberta_cv(train, test, n_folds=CFG['N_FOLDS'])
print(f"\nTraining complete. test_deb shape={test_deb.shape}, oof_deb shape={oof_deb.shape}")


[DeBERTa] === fold 0 (train=1500, val=500) ===


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    step 0/1500  loss=0.4200  elapsed=0s
    step 25/1500  loss=0.5001  elapsed=5s
    step 50/1500  loss=0.5524  elapsed=10s
    step 75/1500  loss=0.4661  elapsed=16s
    step 100/1500  loss=0.1870  elapsed=21s
    step 125/1500  loss=0.6839  elapsed=26s
    step 150/1500  loss=0.2254  elapsed=31s
    step 175/1500  loss=0.5252  elapsed=36s
    step 200/1500  loss=0.4534  elapsed=41s
    step 225/1500  loss=0.5627  elapsed=47s
    step 250/1500  loss=0.3564  elapsed=52s
    step 275/1500  loss=0.5522  elapsed=57s
    step 300/1500  loss=0.3618  elapsed=62s
    step 325/1500  loss=0.4312  elapsed=67s
    step 350/1500  loss=0.7659  elapsed=72s
    step 375/1500  loss=0.6025  elapsed=78s
    step 400/1500  loss=0.5380  elapsed=83s
    step 425/1500  loss=0.2980  elapsed=88s
    step 450/1500  loss=0.2655  elapsed=93s
    step 475/1500  loss=0.6535  elapsed=98s
    step 500/1500  loss=0.2859  elapsed=103s
    step 525/1500  loss=0.8056  elapsed=108s
    step 550/1500  loss=0.4401  elaps

deb/best_val_map3,▁
deb/epoch,▁█
deb/fold,▁▁
deb/train_loss,█▁
deb/val_acc,▁█
deb/val_f1,▁█
deb/val_loss,█▁
deb/val_map3,▁█
deb/best_val_map3,0.97233
deb/epoch,1
deb/fold,0


...


Uploading: 100%|██████████| 10.1k/10.1k [00:00<00:00, 39.2kB/s]

Upload successful: /teamspace/studios/this_studio/artifacts/deberta/test_deb.npy (10KB)
Starting upload for file /teamspace/studios/this_studio/artifacts/deberta/best_deberta_fold4.pth



Uploading: 100%|██████████| 1.74G/1.74G [00:15<00:00, 109MB/s] 

Upload successful: /teamspace/studios/this_studio/artifacts/deberta/best_deberta_fold4.pth (2GB)


Your model instance has been created.
Files are being processed...
See at: https://kaggle.com/models/kamal134134/genai_project/pyTorch/deberta_rag_l_4_2
  [hub] upload complete

Training complete. test_deb shape=(500, 5), oof_deb shape=(2000, 5)


In [9]:
import numpy as np
import pandas as pd
from scipy.special import softmax

TEST_CSV_PATH  = '/teamspace/studios/this_studio/test.csv'
NPY_RAG_PATH   = 'artifacts/deberta/test_deb.npy'
NPY_NORAG_PATH = '/teamspace/studios/this_studio/test_debert (1).npy'

OPTS = ['A', 'B', 'C', 'D', 'E']

W_RAG   = 0.9
W_NORAG = 0.1
T_RAG   = 0.8
T_NORAG = 1.0
POWER   = 2.5

test_df = pd.read_csv(TEST_CSV_PATH)

logits_rag   = np.load(NPY_RAG_PATH)
logits_norag = np.load(NPY_NORAG_PATH)

assert len(test_df) == len(logits_rag) == len(logits_norag), "Rows mismatch! Check your files."

probs_rag   = softmax(logits_rag / T_RAG, axis=1)
probs_norag = softmax(logits_norag / T_NORAG, axis=1)

blended_probs = (W_RAG * (probs_rag ** POWER)) + (W_NORAG * (probs_norag ** POWER))
blended_probs = blended_probs ** (1.0 / POWER)
blended_probs = blended_probs / np.sum(blended_probs, axis=1, keepdims=True)

predictions_strings = []
for i in range(len(blended_probs)):
    top3_idx = np.argsort(-blended_probs[i])[:3]
    top3_letters = [OPTS[idx] for idx in top3_idx]
    predictions_strings.append(" ".join(top3_letters))

sub = pd.DataFrame({
    'id': test_df['id'],
    'prediction': predictions_strings
})

sub.to_csv('submission.csv', index=False)
print("Submission file ready.")
print(sub.head())

 Submission File Ready! 
   id prediction
0   1      A E C
1   2      B E A
2   3      B E D
3   4      E B A
4   5      C D A


In [46]:
push_model(
    "deberta",
    "/teamspace/studios/this_studio/upload_model",
    "v2"
)

  [hub] uploading 'deberta' ...
Uploading Model https://kaggle.com/models/kamal134134/genai_project/pyTorch/deberta_rag_l_4_2 ...
Starting upload for file /teamspace/studios/this_studio/upload_model/best_deberta_fold1.pth


Uploading: 100%|██████████| 1.74G/1.74G [00:15<00:00, 111MB/s] 

Upload successful: /teamspace/studios/this_studio/upload_model/best_deberta_fold1.pth (2GB)
Starting upload for file /teamspace/studios/this_studio/upload_model/best_deberta_fold0.pth



Uploading: 100%|██████████| 1.74G/1.74G [00:17<00:00, 100MB/s] 

Upload successful: /teamspace/studios/this_studio/upload_model/best_deberta_fold0.pth (2GB)
Starting upload for file /teamspace/studios/this_studio/upload_model/oof_deb.npy



Uploading: 100%|██████████| 40.1k/40.1k [00:00<00:00, 149kB/s]

Upload successful: /teamspace/studios/this_studio/upload_model/oof_deb.npy (39KB)
Starting upload for file /teamspace/studios/this_studio/upload_model/test_deb.npy



Uploading: 100%|██████████| 10.1k/10.1k [00:00<00:00, 41.1kB/s]

Upload successful: /teamspace/studios/this_studio/upload_model/test_deb.npy (10KB)


Your model instance version has been created.
Files are being processed...
See at: https://kaggle.com/models/kamal134134/genai_project/pyTorch/deberta_rag_l_4_2
  [hub] upload complete
